# txt2reward — LLM-as-Reward for Highway-RL

PPO on `highway-v0` with a Groq LLM (llama-3.3-70b) as the reward signal.

> **Make sure the GPU runtime is enabled** — Runtime → Change runtime type → T4 GPU

## 1. Install dependencies

In [ ]:
%%capture
!pip install gymnasium highway-env stable-baselines3 groq numpy

## 2. Set GROQ_API_KEY

Get your key at [console.groq.com](https://console.groq.com) and paste it below.

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxx"  # <- paste your key here

# Quick connectivity check
from groq import Groq
_ = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client OK")

## 3. Clone the repository

In [ ]:
!git clone https://github.com/A-Nematkhah/txt2reward.git
%cd txt2reward

## 4. Pre-fill the reward cache

Queries Groq for all **192 possible states** and saves them to `reward_cache.pkl`.
Training will then run completely offline — no HTTP calls in the PPO loop.

⏱ Takes about **2 minutes** on the Groq free tier.

In [ ]:
from llm_judge import prefill_cache_sync
prefill_cache_sync(verbose=True)

## 5. Sanity-check the LLM judge

In [ ]:
from llm_judge import judge_state, reward_cache

test_cases = [
    (30.0, 2, 25.0, "expected: positive reward"),
    (10.0, 0,  3.0, "expected: negative reward"),
    (40.0, 3,  8.0, "expected: negative reward"),
]

for speed, lane, dist, label in test_cases:
    r = judge_state(speed, lane, dist)
    print(f"speed={speed:4.0f}  lane={lane}  dist={dist:4.0f}m  reward={r:+.3f}  ({label})")

print(f"
Cache size: {len(reward_cache)} entries")

## 6. Train

-  — four parallel environments
-  — skip pre-fill since the cache is already ready

Expected time on T4: **~5–10 minutes**

In [ ]:
!python train.py --timesteps 100000 --n-envs 4 --no-precompute

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

## 8. Evaluate the trained model

In [ ]:
import gymnasium as gym
import highway_env
from stable_baselines3 import PPO
from train import make_env

env   = make_env(rank=0, llm_interval=50)()
model = PPO.load("ppo_highway_qwen_reward", env=env)

n_episodes = 5
rewards    = []
for ep in range(n_episodes):
    obs, _ = env.reset()
    done, ep_r = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(action)
        ep_r += r
        done  = terminated or truncated
    rewards.append(ep_r)
    print(f"Episode {ep + 1:2d}: total reward = {ep_r:.2f}")

print(f"
Mean reward over {n_episodes} episodes: {sum(rewards) / n_episodes:.2f}")
env.close()